# Engagement 2 · Mobile-money agent network: audit, clean, merge, summarise

**Client question:** which agents and counties handle the most deposits and withdrawals?
**Problem:** the legacy transaction extract is unreliable, so it is audited and cleaned *before* any analysis, and every change is logged for the data governance team.

Cleaning rules live in `cleaning_pipeline.py` (`clean_agent_data`, `audit_agent_data`); this notebook calls them and explains the decisions.

In [1]:
import os
import pandas as pd
from cleaning_pipeline import clean_agent_data, audit_agent_data

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:,.0f}".format)

## 1. Load and first look
Two reads of the same file: with pandas defaults (what an analyst would normally do; the text `N/A` silently becomes `NaN`) and as untouched text (`dtype=str, keep_default_na=False`), which lets the audit tell blanks from `N/A`.

In [2]:
txns = pd.read_csv("agent_transactions.csv")        # pandas defaults
agents = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)   # untouched text

print("transactions:", txns.shape, "| agents:", agents.shape)
print("\nMissing values per column (pandas defaults):")
print(txns.isna().sum().to_string())
print("\nIn the untouched text: blank amounts =", (txns_raw.amount == "").sum(),
      '| the literal text "N/A" =', (txns_raw.amount == "N/A").sum(),
      "| total isna() under defaults =", txns.amount.isna().sum())
txns.head()

transactions: (612, 5) | agents: (20, 3)

Missing values per column (pandas defaults):
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0

In the untouched text: blank amounts = 15 | the literal text "N/A" = 7 | total isna() under defaults = 22


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353,2026-03-03
1,TX00002,AG020,deposit,"1,238",2026-04-27
2,TX00003,AG008,DEPOSIT,"9,129",2026-04-29
3,TX00004,AG009,deposit,652,2026-04-24
4,TX00005,AG009,withdrawal,478,2026-04-17


## 2. Data-quality audit
One row per issue, tagged with the quality dimension it violates. Counts are **raw rows, duplicates included**; rows with `0` are checks that passed.

In [3]:
audit = audit_agent_data(txns_raw, agents)
audit

,issue,dimension,column,rows_affected
0,Repeated txn_id (duplicate transactions),Uniqueness,txn_id,12
1,Duplicate agent_id in agent register,Uniqueness,agent_id (agents),0
2,txn_type has mixed case / trailing spaces (not exactly Deposit or Withdrawal),Consistency,txn_type,384
3,txn_date uses '/' instead of '-' (mixed formats),Consistency,txn_date,5
4,agent_id not in agent register (orphan key),Consistency,agent_id,8
5,amount blank (missing),Completeness,amount,15
6,amount is non-numeric text such as 'N/A',Validity,amount,7
7,amount <= 0,Validity,amount,0
8,txn_date cannot be parsed even after normalising separators,Validity,txn_date,0
9,"amount outside IQR fences (> 6,044)",Accuracy,amount,40


## 3. Run the cleaning pipeline
All rules run inside one function; the returned log records what each rule changed.

In [4]:
clean, log = clean_agent_data(txns, agents)
log

,step,dimension,rule,rows_affected,rows_after,detail
0,0,-,Raw input,0,612,612 rows loaded
1,1,Uniqueness,"Drop repeated txn_id, keeping the copy with a registered agent",12,600,11 were exact full-row copies; 1 differed only in agent_id
2,2,Consistency,"Trim, lower-case and map txn_type to Deposit/Withdrawal",376,600,0 values could not be mapped
3,3,Validity,Normalise '/' dates and convert txn_date to datetime,5,600,0 dates unparseable (left as NaT)
4,4,Validity,Convert amount to numeric (blank / 'N/A' / non-positive -> NaN),20,600,0 non-positive values treated as invalid
5,5,Accuracy,"Flag IQR outliers (k=1.5); hold out amount == 250,000",40,600,"fences [-1,928, 6,084]; 5 rows held for review; rows are flagged, not deleted"
6,6,Completeness,Impute missing/invalid amount with per-type median; set amount_imputed,20,600,"medians: Deposit=1,796, Withdrawal=1,746"
7,7,Consistency,Left-join agent register; unmatched agents get county='Unknown',7,600,kept (not dropped) and flagged with agent_known=False


## 4. Uniqueness: duplicate transactions

In [5]:
rows_raw, rows_clean = len(txns), len(clean)
removed = rows_raw - rows_clean
full_dups = int(txns_raw.duplicated().sum())

dup_rows = txns_raw[txns_raw.duplicated("txn_id", keep=False)]
conflict_ids = dup_rows.groupby("txn_id").agent_id.nunique().loc[lambda s: s > 1].index.tolist()

print(f"raw rows {rows_raw} -> clean rows {rows_clean}; removed {removed} duplicate transactions")
print(f"exact full-row copies: {full_dups}; copies that differ in agent_id: {removed - full_dups} -> {conflict_ids}")
assert clean.txn_id.is_unique
n_orphan_raw = int((~txns_raw.agent_id.isin(agents.agent_id)).sum())
n_orphan_clean = int((~clean.agent_known).sum())
dup_rows[dup_rows.txn_id.isin(conflict_ids)]

raw rows 612 -> clean rows 600; removed 12 duplicate transactions
exact full-row copies: 11; copies that differ in agent_id: 1 -> ['TX00405']


,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03


**Result.** 12 duplicate transactions were removed (612 -> 600 rows). `txn_id` is the business key, so duplicates are identified on `txn_id` rather than on the whole row: 11 were exact copies, and one (`TX00405`) was a copy whose `agent_id` had been overwritten with an unregistered code. For that one the copy carrying a registered agent was kept, which is why unknown-agent rows fall from 8 raw to 7 after de-duplication.

## 5. Consistency: `txn_type`

In [6]:
print("BEFORE (repr shows hidden spaces):")
print(txns_raw.txn_type.map(repr).value_counts().to_string())
print("\nAFTER:")
print(clean.txn_type.value_counts().to_string())
assert set(clean.txn_type) == {"Deposit", "Withdrawal"}
n_variants = txns_raw.txn_type.nunique()
n_nonstandard = int((~txns_raw.txn_type.isin(["Deposit", "Withdrawal"])).sum())

BEFORE (repr shows hidden spaces):
txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100

AFTER:
txn_type
Deposit       373
Withdrawal    227


**Result.** 5 spellings (case differences plus a trailing space in `'DEPOSIT '`) were reduced to exactly two values, `Deposit` and `Withdrawal`. 384 of the 612 raw rows needed changing. The rule is *trim, lower-case, then map*, and anything unmappable would be reported rather than guessed; none were.

## 6. Validity and types: `amount` and `txn_date`

In [7]:
n_slash = int(txns_raw.txn_date.str.contains("/", regex=False).sum())
n_na_text = int((txns_raw.amount == "N/A").sum())
print("BEFORE:", txns.dtypes.astype(str).to_dict())
print("AFTER :", clean[["amount", "txn_date"]].dtypes.astype(str).to_dict())
date_min, date_max = clean.txn_date.min().date(), clean.txn_date.max().date()
print("\nDate range:", clean.txn_date.min().date(), "to", clean.txn_date.max().date(),
      "| unparseable dates:", clean.txn_date.isna().sum())
print("Amount range (all rows):", f"{clean.amount.min():,.0f} to {clean.amount.max():,.0f}")

BEFORE: {'txn_id': 'str', 'agent_id': 'str', 'txn_type': 'str', 'amount': 'float64', 'txn_date': 'str'}
AFTER : {'amount': 'float64', 'txn_date': 'datetime64[us]'}

Date range: 2026-03-01 to 2026-04-29 | unparseable dates: 0
Amount range (all rows): 109 to 250,000


**Result.** In the raw text `amount` is a string column (it contains blanks and `N/A`); with pandas defaults it happens to load as float because both become `NaN`, which hides the `N/A` problem. The pipeline therefore handles either form with `pd.to_numeric(errors="coerce")`: blanks, `N/A` and any non-positive value become `NaN` for the completeness step. 5 dates used `/` instead of `-`; they were normalised and the whole column parsed with a strict `%Y-%m-%d` format, leaving no unparseable dates. Dates are exported in ISO format.

## 7. Completeness: missing and invalid amounts

In [8]:
n_imputed = int(clean.amount_imputed.sum())
share_rows = n_imputed / len(clean)
share_value = clean.loc[clean.amount_imputed, "amount"].sum() / clean.amount.sum()
meds = clean[~clean.amount_imputed & ~clean.hold_for_review].groupby("txn_type").amount.median()
tot_all = clean[~clean.hold_for_review].groupby("county").amount.sum()
tot_obs = clean[~clean.hold_for_review & ~clean.amount_imputed].groupby("county").amount.sum()
rank_same = tot_all.drop("Unknown").sort_values(ascending=False).index.tolist() == tot_obs.drop("Unknown").sort_values(ascending=False).index.tolist()
rank_msg = "is unchanged" if rank_same else "changes"
print(f"imputed rows: {n_imputed} ({share_rows:.1%} of rows, {share_value:.1%} of total value)")
print("per-type medians used:\n" + meds.to_string())
clean.groupby("txn_type").amount_imputed.agg(["sum", "count"]).rename(columns={"sum": "imputed", "count": "rows"})

imputed rows: 20 (3.3% of rows, 1.3% of total value)
per-type medians used:
txn_type
Deposit      1,796
Withdrawal   1,746


,imputed,rows
txn_type,,
Deposit,13,373
Withdrawal,7,227


**Decision: impute with the median amount of the same `txn_type`, and flag every imputed row with `amount_imputed = True`.**

* *Why not drop the rows?* 20 transactions (3.3%) are real events that moved between agents; dropping them would understate transaction counts for exactly the agents with the worst data.
* *Why not the mean?* Amounts are strongly right-skewed (a few large values), so the mean overstates a typical transaction. The median is robust, and it is calculated on observed values only, excluding the held-out KES 250,000 rows.
* *Why not leave `NaN`?* Sums and averages would silently skip these rows and every county total would be understated by an unknown amount.

**Caveat.** Imputed amounts are estimates, so they must never be used for reconciliation or customer-level figures. They make up 1.3% of total value, and the county ranking by total value is unchanged when they are excluded; the flag lets anyone re-run a summary without them. The proper fix is to recover the true amounts from the source system.

## 8. Outliers: IQR method and the KES 250,000 values

In [9]:
valid = clean.loc[~clean.amount_imputed, "amount"]
q1, q3 = valid.quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
n_outliers = int(clean.is_outlier.sum())
n_hold = int(clean.hold_for_review.sum())
n_natural = n_outliers - n_hold
others = clean.loc[~clean.hold_for_review & ~clean.amount_imputed, "amount"]
max_other = others.max()
ratio = 250_000 / max_other
print(f"Q1={q1:,.0f}  Q3={q3:,.0f}  fences=[{lower:,.0f}, {upper:,.0f}]")
print(f"IQR outliers: {n_outliers}  (of which KES 250,000: {n_hold}; other large values: {n_natural})")
print(f"largest non-250,000 amount: {max_other:,.0f}  -> 250,000 is {ratio:.1f}x larger")
hold = clean[clean.hold_for_review]
hold_agents, hold_counties = hold.agent_id.nunique(), hold.county.nunique()
hold_value = hold.amount.sum()
hold_dates = hold.txn_date.nunique()
hold_share = hold_value / clean.amount.sum()
hold[["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date"]]

Q1=1,077  Q3=3,080  fences=[-1,928, 6,084]
IQR outliers: 40  (of which KES 250,000: 5; other large values: 35)
largest non-250,000 amount: 28,847  -> 250,000 is 8.7x larger


,txn_id,agent_id,county,txn_type,amount,txn_date
306,TX00307,AG012,Kisumu,Deposit,"250,000",2026-03-02
315,TX00316,AG014,Nairobi,Deposit,"250,000",2026-04-01
424,TX00425,AG005,Mombasa,Withdrawal,"250,000",2026-03-02
463,TX00464,AG009,Nakuru,Withdrawal,"250,000",2026-04-11
594,TX00595,AG016,Kiambu,Deposit,"250,000",2026-03-11


**Findings.** The IQR rule (Q1 - 1.5 x IQR, Q3 + 1.5 x IQR) flags 40 transactions above KES 6,084. Of these, 35 are ordinary large transactions sitting in the long tail of a skewed distribution, the biggest being KES 28,847. The other 5 are the KES 250,000 values, and they behave differently:

* the **identical round amount** appears 5 times in a column of otherwise irregular figures, which is very unlikely by chance;
* each is **8.7 times larger** than the largest other transaction;
* they are spread over **5 different agents in 5 counties** and 4 different dates, which points to a system-level cause (a default or placeholder value, a capped or limit amount, a failed-load fill) rather than one agent's behaviour.

**Recommendation.**
1. **Do not delete and do not silently accept.** Keep the rows, flagged `is_outlier` and `hold_for_review`.
2. **Hold them out of totals and averages** until verified. Five rows would otherwise add KES 1,250,000 (46% of all value), lifting every county by exactly KES 250,000 and masking real differences between counties; summaries below exclude them and show the effect.
3. **Verify against the source ledger / settlement records** and, where needed, with the agents. If genuine, restore them; if a placeholder or system default, correct or void them. KES 250,000 may coincide with a maximum single-transaction limit, so check the current limit in the product rules.
4. Add a **validation rule at source** (round-value and limit-hit alerts). The 35 other outliers look like genuine large transactions: keep them, and leave any threshold monitoring to compliance.

## 9. Merge with the agent register and post-merge validation

In [10]:
pre_merge_rows = int(log.loc[log.step == 6, "rows_after"].iloc[0])
post_merge_rows = len(clean)
unknown = clean[~clean.agent_known]
n_unknown = len(unknown)
unknown_value = unknown.amount.sum()
unknown_share = n_unknown / len(clean)
print(f"rows before merge: {pre_merge_rows} | after merge: {post_merge_rows} | difference: {post_merge_rows - pre_merge_rows}")
print(f"agent_id unique in register: {agents.agent_id.is_unique} | txn_id unique after merge: {clean.txn_id.is_unique}")
print(f"transactions with unknown agents: {n_unknown} ({n_unknown / len(clean):.1%}) worth KES {unknown_value:,.0f}")
print("unknown agent ids:", unknown.agent_id.value_counts().to_dict())
print("registered agents with no transactions:", sorted(set(agents.agent_id) - set(clean.agent_id)))
clean.county.value_counts().rename("transactions").to_frame()

rows before merge: 600 | after merge: 600 | difference: 0
agent_id unique in register: True | txn_id unique after merge: True
transactions with unknown agents: 7 (1.2%) worth KES 27,156
unknown agent ids: {'AG099': 7}
registered agents with no transactions: []


,transactions
county,
Nairobi,180
Mombasa,133
Kisumu,111
Nakuru,99
Kiambu,70
Unknown,7


**Result.** The left join kept all 600 rows (no row lost or duplicated, enforced with `validate="m:1"`). **7 of 600 transactions (1.2%) reference an agent that is not in the register** (`AG099`), together worth KES 27,156.

**Handling.** They are kept, not dropped: the money moved, and deleting them would understate network activity. They receive `county = "Unknown"` and `agent_known = False`, so they appear as their own line in the summaries rather than being credited to a real county. The data governance team should establish whether `AG099` is an unregistered agent, a test account or a mistyped ID; until then these rows cannot be attributed to a county.

## 10. Summaries

Both summaries exclude the 5 held-out KES 250,000 rows (`hold_for_review`) and include imputed amounts (flagged). The comparison table shows how much the held-out rows would have changed each county.

In [11]:
base = clean[~clean.hold_for_review].copy()

county_summary = (base.groupby(["county", "txn_type"])
                      .agg(total_value=("amount", "sum"),
                           n_transactions=("txn_id", "count"),
                           avg_value=("amount", "mean"))
                      .reset_index())
county_summary

,county,txn_type,total_value,n_transactions,avg_value
0,Kiambu,Deposit,"84,737",42,"2,018"
1,Kiambu,Withdrawal,"77,020",27,"2,853"
2,Kisumu,Deposit,"186,479",73,"2,555"
3,Kisumu,Withdrawal,"76,344",37,"2,063"
4,Mombasa,Deposit,"149,064",72,"2,070"
5,Mombasa,Withdrawal,"118,592",60,"1,977"
6,Nairobi,Deposit,"293,668",117,"2,510"
7,Nairobi,Withdrawal,"161,916",62,"2,612"
8,Nakuru,Deposit,"184,929",60,"3,082"
9,Nakuru,Withdrawal,"78,564",38,"2,067"


In [12]:
by_county = (base.pivot_table(index="county", columns="txn_type", values="amount",
                              aggfunc="sum", margins=True, margins_name="Total")
                 .sort_values("Total", ascending=False))
with_hold = clean.groupby("county").amount.sum().rename("total_incl_held_rows")
by_county.join(with_hold).assign(
    effect_of_held_rows=lambda d: d.total_incl_held_rows - d.Total).drop(index="Total", errors="ignore")

,Deposit,Withdrawal,Total,total_incl_held_rows,effect_of_held_rows
county,,,,,
Nairobi,"293,668","161,916","455,584","705,584","250,000"
Mombasa,"149,064","118,592","267,656","517,656","250,000"
Nakuru,"184,929","78,564","263,494","513,494","250,000"
Kisumu,"186,479","76,344","262,824","512,824","250,000"
Kiambu,"84,737","77,020","161,758","411,758","250,000"
Unknown,"25,279","1,877","27,156","27,156",0


In [13]:
real = by_county.drop(index="Total").drop(index="Unknown", errors="ignore")
top_county, top_total = real.Total.idxmax(), real.Total.max()
second_county = real.Total.nlargest(2).index[1]
dep_leader = real["Deposit"].idxmax()
wd_leader = real["Withdrawal"].idxmax()
agent_tot = (base[base.agent_known].groupby("agent_id").amount.agg(["sum", "count"])
             .sort_values("sum", ascending=False))
top_agent, top_agent_total = agent_tot.index[0], agent_tot["sum"].iloc[0]
lead_text = (f"{dep_leader} leads on both deposits and withdrawals" if dep_leader == wd_leader
             else f"{dep_leader} leads on deposits and {wd_leader} on withdrawals")
n_agents = agents.county.value_counts()
per_agent = (real.Total / n_agents).sort_values(ascending=False).rename("value_per_agent")
top_pa, top_pa_value = per_agent.index[0], per_agent.iloc[0]
pd.concat([real.Total, n_agents.rename("agents"), per_agent], axis=1).sort_values("value_per_agent", ascending=False)

,Total,agents,value_per_agent
county,,,
Nakuru,"263,494",3,"87,831"
Kiambu,"161,758",2,"80,879"
Nairobi,"455,584",6,"75,931"
Kisumu,"262,824",4,"65,706"
Mombasa,"267,656",5,"53,531"


**Answer for the regional manager.** Among registered counties, **Nairobi** handles the most value (KES 455,584), ahead of Mombasa; Nairobi leads on both deposits and withdrawals. The single busiest agent by value is **AG006** (KES 110,037). County totals partly reflect how many agents each county has, so value per agent is a fairer comparison: on that basis **Nakuru** is highest (KES 87,831 per agent, table above). Treat these as provisional until the held rows and the unknown agent are resolved.

### 10(b) Weekly transaction value by county (pivot table)

In [14]:
base["week_commencing"] = (base.txn_date - pd.to_timedelta(base.txn_date.dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")
weekly = base.pivot_table(index="week_commencing", columns="county", values="amount",
                          aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
first_week_days = base[base.week_commencing == weekly.index[0]].txn_date.dt.date.nunique()
last_week_days = base[base.week_commencing == weekly.index[-2]].txn_date.dt.date.nunique()
weekly

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_commencing,,,,,,,
2026-02-23,0,"1,893","6,759","7,046","2,763",0,"18,461"
2026-03-02,"18,317","37,933","35,689","68,967","56,574",0,"217,480"
2026-03-09,"17,742","23,186","30,497","52,880","28,730",0,"153,034"
2026-03-16,"22,054","28,114","25,164","43,127","39,795","1,639","159,894"
2026-03-23,"11,504","27,077","17,050","41,393","18,968",0,"115,992"
2026-03-30,"22,609","41,077","38,344","32,931","18,145","8,507","161,613"
2026-04-06,"15,030","15,341","47,112","56,806","26,609","9,824","170,722"
2026-04-13,"12,265","29,455","26,730","56,402","26,172",0,"151,024"
2026-04-20,"24,507","37,439","16,687","65,127","30,862","1,877","176,499"


Weeks start on Monday. The first and last weeks are partial (the extract starts on a Sunday, 1 March, and stops part-way through a week), so their totals are lower for calendar reasons, not because activity fell. `Unknown` is the unregistered agent.

## 11. Export the clean file

In [15]:
os.makedirs("outputs", exist_ok=True)
out = clean.assign(txn_date=clean.txn_date.dt.strftime("%Y-%m-%d"),
                   agent_onboarded=clean.agent_onboarded.dt.strftime("%Y-%m-%d"))
out.to_csv("outputs/agent_transactions_clean.csv", index=False)

chk = pd.read_csv("outputs/agent_transactions_clean.csv")
assert chk.txn_id.is_unique
assert set(chk.txn_type) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_numeric_dtype(chk.amount) and chk.amount.notna().all()
assert chk.txn_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert {"amount_imputed", "is_outlier", "county"} <= set(chk.columns)
print(f"saved {len(chk)} rows (raw extract had {len(txns)}); all output checks passed")
pd.DataFrame({"stage": ["raw extract", "after removing duplicates / clean"], "rows": [len(txns), len(chk)]})

saved 600 rows (raw extract had 612); all output checks passed


,stage,rows
0,raw extract,612
1,after removing duplicates / clean,600


# Data Quality Report · agent transaction extract

**Scope.** `agent_transactions.csv` (612 rows, March-April 2026) joined to `agents.csv` (20 agents). After cleaning: **600 unique transactions**. Every rule below is in `cleaning_pipeline.py` and recorded in the cleaning log.

| Dimension | Finding | Action taken | Residual risk |
|---|---|---|---|
| **Accuracy** | 5 transactions of exactly KES 250,000, 8.7x the next-largest value, across 5 agents; 35 other IQR outliers look genuine | Flagged (`is_outlier`), the 5 held out of totals (`hold_for_review`), none deleted | High until verified against the source ledger |
| **Completeness** | 20 unique transactions with a missing amount (blank or `N/A`) | Median of same `txn_type` imputed, `amount_imputed` flag | Estimates, about 1.3% of total value; not for reconciliation |
| **Consistency** | 5 spellings of `txn_type` (384 raw rows); 5 dates with `/`; 7 transactions for agent `AG099`, absent from the register | Mapped to Deposit/Withdrawal; dates normalised; unknown agents kept with county `Unknown` and `agent_known=False` | `AG099` unattributed to any county |
| **Timeliness** | Dates run 2026-03-01 to 2026-04-29; none in the future or before the agent's onboarding date | None needed | Extract covers only about two months; first and last weeks are partial |
| **Uniqueness** | 12 repeated `txn_id` rows (11 exact copies, 1 with a conflicting `agent_id`) | Removed on `txn_id`, keeping the copy with a registered agent | Low; register has no duplicate agents |
| **Validity** | `amount` is text in the raw file, including `N/A` entries; mixed date formats | Converted to numeric and `datetime`; invalid amounts handled under Completeness; 0 unparseable dates | Low |

**Recommendations.** (1) Reconcile the KES 250,000 rows with the source system before any figure is published. (2) Identify `AG099`. (3) Fix the legacy export (typed numeric fields, controlled values for `txn_type`, ISO dates, unique `txn_id` constraint, foreign key to the agent register).

**Privacy: personal data if customer phone numbers were included.** A mobile number identifies a natural person, so it would be personal data under the Kenya Data Protection Act 2019, and combined with `txn_id`, `amount`, `txn_date` and agent location it would reveal that person's financial behaviour and movements. In the current extract `agent_id` could also be personal data where an agent is an individual sole trader. To protect such fields I would: collect and keep only what the analysis needs (data minimisation), so the phone number is dropped from analytical extracts; where a customer link is required, replace it with a keyed hash or token with the key held separately and access restricted; mask any display value (for example, show only the last three digits); store the extract encrypted with role-based access and an audit trail, and keep it out of the code repository; set a retention period and delete the data when the purpose ends; process it only for the stated purpose, with a documented lawful basis, and carry out a data protection impact assessment because financial transaction data is high-risk; avoid transfer outside Kenya unless the Act's conditions are met; and make sure the provider is registered with the Office of the Data Protection Commissioner and can honour data-subject rights. This is a practical summary, not legal advice; the provider's data protection officer or counsel should confirm the position.